In [5]:
import os
import json
import openai
openai.api_key = 'sk-proj-vbjhDtqmIksUSo6rDwMVT3BlbkFJFdA2SoSwH7Gkv5iobzvO'


In [6]:

def analyze_bias(text, original_text):
    prompt = (
        f"Analyze the following text for different types of biases and compare it to the original text:\n\n"
        f"Original Text: {original_text}\n\n"
        f"Text: {text}\n\n"
        f"1. Sentiment: Provide a summary of the sentiment, notable examples of biased language, and rate the bias from 0 (no bias) to 1 (extreme bias). Describe the sentiment bias in three words.\n"
        f"2. Framing: Describe how the text frames the topic, identify any bias in the framing, provide notable examples, and rate the bias from 0 (no bias) to 1 (extreme bias). Describe the framing bias in three words.\n"
        f"3. Representation: Examine the representation of different groups in the text, identify any biases in how these groups are portrayed, provide notable examples, and rate the representation bias from 0 (no bias) to 1 (extreme bias). Describe the representation bias in three words.\n"
        f"4. Omission: Identify any significant omissions or differences in information that might indicate bias in the text, provide notable examples, and rate the omission bias from 0 (no bias) to 1 (extreme bias). Describe the omission bias in three words.\n"
        f"Additionally, extract any dates and times mentioned in the text along with a brief description of the associated events."
    )

    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=1500,
        n=1,
        stop=None,
        temperature=0.7,
    )

    return response.choices[0].message['content'].strip()

def extract_bias_scores_and_examples(analysis_text):
    lines = analysis_text.split('\n')
    scores = {
        "sentiment": {"score": 0.0, "examples": [], "description": ""},
        "framing": {"score": 0.0, "examples": [], "description": ""},
        "representation": {"score": 0.0, "examples": [], "description": ""},
        "omission": {"score": 0.0, "examples": [], "description": ""},
        "events": []
    }
    current_section = None

    for line in lines:
        line = line.strip()
        if line.startswith("1. Sentiment:"):
            current_section = "sentiment"
        elif line.startswith("2. Framing:"):
            current_section = "framing"
        elif line.startswith("3. Representation:"):
            current_section = "representation"
        elif line.startswith("4. Omission:"):
            current_section = "omission"
        elif line.startswith("Additionally, extract any dates and times mentioned"):
            current_section = "events"
        elif current_section in scores:
            if 'rate the bias from 0 (no bias) to 1 (extreme bias)' in line:
                try:
                    score = float(line.split(':')[-1].strip())
                    score = max(0.0, min(1.0, score))  # Ensure the score is between 0 and 1
                    scores[current_section]["score"] = score
                except ValueError:
                    continue
            elif 'Describe the' in line and current_section:
                scores[current_section]["description"] = line.split(':')[-1].strip()
            elif 'example' in line.lower() or 'notable' in line.lower():
                scores[current_section]["examples"].append(line)
        elif current_section == "events":
            if line:
                scores["events"].append(line)
    
    return scores

def process_documents(folder_path):
    results = {}

    for filename in os.listdir(folder_path):
        if filename.endswith('.txt') and '_1_' in filename:
            file_path = os.path.join(folder_path, filename)
            original_file_path = file_path.replace('_1_', '_0_')
            
            if os.path.exists(original_file_path):
                with open(file_path, 'r', encoding='utf-8') as file:
                    text_variant = file.read()
                with open(original_file_path, 'r', encoding='utf-8') as original_file:
                    original_text = original_file.read()

                analysis = analyze_bias(text_variant, original_text)
                scores_and_examples = extract_bias_scores_and_examples(analysis)

                results[filename] = scores_and_examples

    return results

def main():
    folder_path = r"C:\Users\Raphael\Projects\VAST2024\MC1\mc1_data\articles"
    
    results = process_documents(folder_path)
    
    output_file = r"C:\Users\Raphael\Projects\VAST2024\MC1\mc1_data\analysis_results.json"
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(results, file, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    main()


APIConnectionError: Error communicating with OpenAI: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))